In [ ]:
!pip install -q langchain-text-splitters chromadb sentence-transformers

In [ ]:
import os

os.makedirs("docs", exist_ok=True)

article_text = """Artificial Intelligence: An Overview

Artificial Intelligence, or AI, is the field of computer science focused on building systems
that can perform tasks which normally require human intelligence. These tasks include
understanding language, recognizing images, making decisions, and learning from experience.

Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardcoded rules. Instead of programming every decision by hand, we feed the system many examples,
and it learns the underlying pattern on its own. This approach powers everything from spam filters
to recommendation systems.

Deep Learning takes Machine Learning further by using neural networks with many layers. These
layered networks can automatically discover complex patterns in large amounts of data, such as
recognizing faces in photos or understanding the meaning of a sentence.

Natural Language Processing, or NLP, is the branch of AI that focuses specifically on human
language. NLP powers chatbots, translation tools, and search engines. A key building block of
modern NLP is the embedding, which converts words or sentences into numeric vectors that capture
meaning.

Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a
language model. Instead of relying only on what a language model memorized during training, RAG
first retrieves relevant, up-to-date information from a document collection, then passes that
information to the model so it can generate a more accurate, grounded answer.

Vector Databases store embeddings and allow fast similarity search across millions of documents.
They are a core infrastructure piece behind modern RAG systems, chatbots, and recommendation
engines used by companies around the world today.
"""

with open("docs/ai_overview.txt", "w") as f:
    f.write(article_text)

print("Document created")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=30,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_text(article_text)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i} ({len(chunk)} chars) ---")
    print(chunk)

In [ ]:
chunk_metadatas = [
    {
        "source": "ai_overview.txt",
        "chunk_index": i,
        "char_count": len(chunk),
        "chunk_size": 150,
        "version": "v2"
    }
    for i, chunk in enumerate(chunks)
]

chunk_ids = [
    f"ai_overview_v2_chunk_{i}"
    for i in range(len(chunks))
]

print("Example ID:", chunk_ids[0])
print("Example metadata:", chunk_metadatas[0])

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.Client()

hf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = client.create_collection(
    name="rag_chunks_150",
    embedding_function=hf_embedder
)

print("Collection created:", collection.name)

In [ ]:
collection.add(
    documents=chunks,
    metadatas=chunk_metadatas,
    ids=chunk_ids
)

print("Total chunks stored:", collection.count())

In [ ]:
stored = collection.get(
    ids=chunk_ids,
    include=["documents", "metadatas"]
)

for cid, doc, meta in zip(
    stored["ids"],
    stored["documents"],
    stored["metadatas"]
):
    print("ID:", cid)
    print("Metadata:", meta)
    print("Text:", doc)
    print("-" * 60)

In [ ]:
results = collection.query(
    query_texts=["How does RAG combine search with language models?"],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

for doc, meta, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print(
        f"Distance: {distance:.4f} | "
        f"Source: {meta['source']} | "
        f"Chunk: {meta['chunk_index']}"
    )
    print(doc)
    print("-" * 60)

In [ ]:
def add_document(collection, splitter, text, source_name):
    chunks = splitter.split_text(text)

    metadatas = [
        {
            "source": source_name,
            "chunk_index": i,
            "char_count": len(chunk),
            "chunk_size": 150
        }
        for i, chunk in enumerate(chunks)
    ]

    ids = [
        f"{source_name.replace('.txt', '')}_v2_chunk_{i}"
        for i in range(len(chunks))
    ]

    collection.add(
        documents=chunks,
        metadatas=metadatas,
        ids=ids
    )

    return chunks, ids

In [ ]:
third_doc_text = """Large Language Models and RAG

Large Language Models are AI systems trained on large collections of text.
RAG improves LLM responses by retrieving relevant information from external documents.

The retrieved information is provided to the language model as context.
This helps generate more accurate and grounded answers.
"""

third_chunks, third_ids = add_document(
    collection,
    splitter,
    third_doc_text,
    "llm_and_rag.txt"
)

print("Third document chunks:", len(third_chunks))
print("Total chunks:", collection.count())